# Investment

##### Import Libraries

In [1]:
import os
import io
import glob
import pandas as pd
import numpy as np
import yfinance as yf
import requests

# Handle Colab-specific imports safely
try:
    from google.colab import drive, auth
    from google.auth import default
    import gspread

    # Mount drive or authenticate only if running in Colab
    drive.mount('/content/drive')
except ImportError:
    print("Running outside Google Colab (e.g. GitHub Actions). Skipping Colab auth.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


##### Account Balance

In [2]:
# 1. Authenticate your Google Account in the notebook
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

# 2. Open the spreadsheet using its URL or ID
sheet_id = '17B8So56dDcPufRE9P7P5uqAkh13Ts0qsFB2DQJo9qUo'
spreadsheet = gc.open_by_key(sheet_id)

# 3. Load the first sheet into a DataFrame
worksheet = spreadsheet.worksheet('investment')
data = worksheet.get_all_values()

# 4. Convert to Pandas DataFrame
headers = data[0]
investment_summary = pd.DataFrame(data[1:], columns=headers)

investment_summary.head()

,month_year,Brokers,Deposit(SGD),Withdraw(SGD),ending_total_assets,ending_cash_balance
0,2022-12-01,Tiger,10000,0,10036.6,0.79
1,2022-12-01,Moomoo,30980,0,16458.95,0
2,2022-12-01,Crypto,34200,0,24964,0
3,2023-01-01,Tiger,5000,0,14808.76,0.01
4,2023-01-01,Crypto,0,0,24964,0


In [3]:
# Specify the path to the folder holding your files
folder_path = '/content/drive/MyDrive/Investment/IB'

csv_files = glob.glob(os.path.join(folder_path, '*.csv'))

# Dictionaries to group tables by index across all files
tables_by_index = {} # e.g., {0: [df_file1, df_file2], 1: [df_file1, df_file2]}

for file_path in csv_files:
    file_name = os.path.basename(file_path)
    print(f"Processing: {file_name}")

    with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
        lines = f.readlines()

    header_indices = []
    for idx, line in enumerate(lines):
        first_col = line.split(',')[0].strip().replace('"', '')
        if first_col == 'CurrencyPrimary':
            header_indices.append(idx)

    if not header_indices:
        print(f"  --> No 'CurrencyPrimary' header found in {file_name}. Skipping.\n")
        continue

    for i in range(len(header_indices)):
        start_idx = header_indices[i]
        end_idx = header_indices[i+1] if i + 1 < len(header_indices) else len(lines)

        table_chunk = "".join(lines[start_idx:end_idx])

        try:
            df = pd.read_csv(io.StringIO(table_chunk))
            df = df.dropna(how='all')

            # Group dataframes by table index (0 for Table 1, 1 for Table 2, etc.)
            if i not in tables_by_index:
                tables_by_index[i] = []
            tables_by_index[i].append(df)

            print(f"  --> Table {i+1} extracted successfully (Shape: {df.shape})")
        except Exception as e:
            print(f"  --> Error parsing Table {i+1}: {e}")

    print("-" * 50)

# Combine matching tables across all files
combined_tables = {}
for table_idx, df_list in tables_by_index.items():
    combined_df = pd.concat(df_list, ignore_index=True)
    combined_tables[f"Table_{table_idx + 1}"] = combined_df
    print(f"Combined Table {table_idx + 1} shape: {combined_df.shape}")

print("\nAll files successfully combined!")

Processing: PL.csv
  --> Table 1 extracted successfully (Shape: (260, 8))
  --> Table 2 extracted successfully (Shape: (222, 10))
  --> Table 3 extracted successfully (Shape: (9, 10))
--------------------------------------------------
Processing: PL (1).csv
  --> Table 1 extracted successfully (Shape: (262, 8))
  --> Table 2 extracted successfully (Shape: (153, 10))
  --> Table 3 extracted successfully (Shape: (6, 10))
--------------------------------------------------
Combined Table 1 shape: (522, 8)
Combined Table 2 shape: (375, 10)
Combined Table 3 shape: (15, 10)

All files successfully combined!


In [4]:
# Access individual combined tables
balance_df = combined_tables['Table_1']
trading_df = combined_tables['Table_2']
div_df = combined_tables['Table_3']

In [5]:
balance_df.head()

,CurrencyPrimary,ReportDate,Cash,CashLong,CashShort,Total,TotalLong,TotalShort
0,SGD,20240902,0.000000,0.000000,0.0,0.000000,0.000000,0.0000
1,SGD,20240903,15052.829326,15052.829326,0.0,14830.622326,15052.829326,-222.2070
2,SGD,20240904,20052.695949,20052.695949,0.0,19812.796749,20052.695949,-239.8992
3,SGD,20240905,20052.542364,20052.542364,0.0,19937.492364,20052.542364,-115.0500
4,SGD,20240906,20104.126996,20104.126996,0.0,19793.456896,20104.126996,-310.6701


In [6]:
trading_df.head()

,CurrencyPrimary,AssetClass,Symbol,Description,Expiry,DateTime,Put/Call,Quantity,ClosePrice,FifoPnlRealized
0,USD,STK,KO,COCA-COLA CO/THE,NaN,20241206;162000,NaN,100.0,62.60,0.000000
1,USD,STK,KO,COCA-COLA CO/THE,NaN,20250131;162000,NaN,-100.0,63.35,75.499335
2,USD,STK,LULU,LULULEMON ATHLETICA INC,NaN,20250725;162000,NaN,100.0,216.59,0.000000
3,USD,STK,MSFT,MICROSOFT CORP,NaN,20250328;162000,NaN,100.0,375.39,0.000000
4,USD,STK,MSFT,MICROSOFT CORP,NaN,20250411;162000,NaN,-100.0,387.81,428.496806


In [7]:
div_df.head()

,CurrencyPrimary,AssetClass,Symbol,Description,Expiry,Put/Call,Date/Time,DividendType,Amount,Type
0,USD,STK,NVDA,NVDA(US67066G1040) CASH DIVIDEND USD 0.01 PER ...,NaN,NaN,20241003;202000,NaN,-0.3,Withholding Tax
1,USD,STK,NVDA,NVDA(US67066G1040) CASH DIVIDEND USD 0.01 PER ...,NaN,NaN,20241003;202000,Ordinary Dividend,1.0,Dividends
2,SGD,NaN,NaN,CASH RECEIPTS / ELECTRONIC FUND TRANSFERS,NaN,NaN,20240903,NaN,5000.0,Deposits/Withdrawals
3,SGD,NaN,NaN,CASH RECEIPTS / ELECTRONIC FUND TRANSFERS,NaN,NaN,20240903,NaN,9990.0,Deposits/Withdrawals
4,SGD,NaN,NaN,CASH RECEIPTS / ELECTRONIC FUND TRANSFERS,NaN,NaN,20240903,NaN,8.0,Deposits/Withdrawals


In [8]:
withdraw_deposit_df = div_df[div_df['Type']=='Deposits/Withdrawals']
withdraw_deposit_df['Type'] = np.where(withdraw_deposit_df['Amount'] > 0, 'Deposit', 'Withdraw')
withdraw_deposit_df.head()

# Clean and convert the 'Date/Time' column to 'YYYY-MM' period format
withdraw_deposit_df['Date/Time'] = withdraw_deposit_df['Date/Time'].astype(str).str.split(';').str[0]
withdraw_deposit_df['Date/Time'] = pd.to_datetime(withdraw_deposit_df['Date/Time'], format='%Y%m%d')
withdraw_deposit_df['Date/Time'] = withdraw_deposit_df['Date/Time'].dt.to_period('M') # Convert to month period

# Pivot the type column to individual Deposit and Withdraw to match investment_summary
pivot_df = withdraw_deposit_df.groupby(['Date/Time', 'CurrencyPrimary']).apply(lambda x:
    pd.Series({
        'Deposit(SGD)': x[x['Type'] == 'Deposit']['Amount'].sum(),
        'Withdraw(SGD)': -x[x['Type'] == 'Withdraw']['Amount'].sum() # Negate withdrawal amounts
    })
).reset_index()

pivot_df = pivot_df.fillna(0)

monthly_pivot_df = pivot_df.groupby('Date/Time').agg({
    'Deposit(SGD)': 'sum',
    'Withdraw(SGD)': 'sum'
}).reset_index()

monthly_pivot_df = monthly_pivot_df.rename(columns={'Date/Time': 'month_year'})

/tmp/ipykernel_12971/2772520589.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  withdraw_deposit_df['Type'] = np.where(withdraw_deposit_df['Amount'] > 0, 'Deposit', 'Withdraw')
/tmp/ipykernel_12971/2772520589.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  withdraw_deposit_df['Date/Time'] = withdraw_deposit_df['Date/Time'].astype(str).str.split(';').str[0]
/tmp/ipykernel_12971/2772520589.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[

In [9]:
# Convert 'ReportDate' to datetime objects
balance_df['ReportDate'] = pd.to_datetime(balance_df['ReportDate'], format='%Y%m%d')

# Extract year and month as Period object
balance_df['YearMonth'] = balance_df['ReportDate'].dt.to_period('M')

# Find the latest date for each month-year
latest_dates_per_month = balance_df.groupby('YearMonth')['ReportDate'].max().reset_index()

# Filter balance_df to keep only rows with 'ReportDate' matching the latest dates per month
# This ensures we get the last day's data for each month
balance_df = balance_df[balance_df['ReportDate'].isin(latest_dates_per_month['ReportDate'])]

# Drop unnecessary columns. Importantly, drop 'ReportDate' as we'll use 'YearMonth' for merging.
balance_df = balance_df.drop(columns=['CurrencyPrimary',
                                      'CashLong', 'CashShort',
                                      'TotalLong', 'TotalShort',
                                      'ReportDate']) # Drop ReportDate here

# Rename 'YearMonth' to 'month_year' for merging
balance_df = balance_df.rename(columns={'YearMonth': 'month_year', # Rename YearMonth to month_year
                                        'Cash' : 'ending_cash_balance',
                                        'Total' : 'ending_total_assets'})

In [10]:
# Merge monthly_pivot_df and balance_df on 'month_year'
merged_ib_account_df = pd.merge(monthly_pivot_df, balance_df, on='month_year', how='outer')

merged_ib_account_df['Brokers'] = 'IB'

# Display the DataFrame with the new 'Brokers' column
merged_ib_account_df.head()

,month_year,Deposit(SGD),Withdraw(SGD),ending_cash_balance,ending_total_assets,Brokers
0,2024-09,20000.00,0.0,21137.118009,21108.910064,IB
1,2024-10,NaN,NaN,21791.462043,21694.945069,IB
2,2024-11,NaN,NaN,22378.297896,22378.297896,IB
3,2024-12,0.02,0.0,14531.016138,22991.654138,IB
4,2025-01,NaN,NaN,15172.214656,23587.942456,IB


In [11]:
investment_summary = pd.concat([investment_summary, merged_ib_account_df], ignore_index=True)
investment_summary.head()

,month_year,Brokers,Deposit(SGD),Withdraw(SGD),ending_total_assets,ending_cash_balance
0,2022-12-01,Tiger,10000,0,10036.6,0.79
1,2022-12-01,Moomoo,30980,0,16458.95,0
2,2022-12-01,Crypto,34200,0,24964,0
3,2023-01-01,Tiger,5000,0,14808.76,0.01
4,2023-01-01,Crypto,0,0,24964,0


In [12]:
# 1. Convert everything to string first, then parse mixed date formats
investment_summary["month_year"] = (
    pd.to_datetime(investment_summary["month_year"].astype(str), format="mixed")
    + pd.offsets.MonthEnd(0)
)

# 2. Convert relevant columns to numeric, handling commas
numeric_cols = ["ending_total_assets", "Deposit(SGD)", "Withdraw(SGD)"]
for col in numeric_cols:
  if col in investment_summary.columns:
    investment_summary[col] = pd.to_numeric(
        investment_summary[col]
        .astype(str)
        .str.replace(",", "", regex=False),
        errors="coerce",
    ).fillna(0)

# 3. Chronological sorting by datetime and Broker
investment_summary = investment_summary.sort_values(by=["month_year", "Brokers"])

# 4. Shift ending_total_assets within Broker group
investment_summary["previous_month_assets"] = (
    investment_summary.groupby("Brokers")["ending_total_assets"]
    .shift(1)
    .fillna(0)
)

# 5. Handle initial row deposits for previous_month_assets
first_row_indices = investment_summary.groupby("Brokers").head(1).index
investment_summary.loc[first_row_indices, "previous_month_assets"] = (
    investment_summary.loc[first_row_indices, "Deposit(SGD)"]
)

# 6. P&L Calculations
full_pnl = (
    investment_summary["ending_total_assets"]
    - investment_summary["previous_month_assets"]
    - investment_summary["Deposit(SGD)"]
    + investment_summary["Withdraw(SGD)"]
)

first_row_pnl = (
    investment_summary["ending_total_assets"]
    - investment_summary["previous_month_assets"]
)

investment_summary["P&L"] = np.where(
    investment_summary.index.isin(first_row_indices), first_row_pnl, full_pnl
)

investment_summary["P&L(percent)"] = np.where(
    investment_summary["previous_month_assets"] != 0,
    investment_summary["P&L"] / investment_summary["previous_month_assets"],
    0,
)

# 7. Cumulative Returns (time weight returns)
investment_summary["Cumulative_R"] = (
    1 + investment_summary["P&L(percent)"]
).groupby(investment_summary["Brokers"]).cumprod() - 1

# 8. Clean up column names for BigQuery compatibility
investment_summary.columns = (
    investment_summary.columns.str.replace(r"[&%()\- ]", "_", regex=True)
    .str.replace(r"_{2,}", "_", regex=True)
    .str.strip("_")
)

investment_summary

,month_year,Brokers,Deposit_SGD,Withdraw_SGD,ending_total_assets,ending_cash_balance,previous_month_assets,P_L,P_L_percent,Cumulative_R
2,2022-12-31,Crypto,34200.0,0.0,24964.000000,0,34200.00000,-9236.000000,-0.270058,-0.270058
1,2022-12-31,Moomoo,30980.0,0.0,16458.950000,0,30980.00000,-14521.050000,-0.468723,-0.468723
0,2022-12-31,Tiger,10000.0,0.0,10036.600000,0.79,10000.00000,36.600000,0.003660,0.003660
4,2023-01-31,Crypto,0.0,0.0,24964.000000,0,24964.00000,0.000000,0.000000,-0.270058
5,2023-01-31,Moomoo,0.0,0.0,20350.000000,0,16458.95000,3891.050000,0.236409,-0.343125
...,...,...,...,...,...,...,...,...,...,...
133,2026-08-31,Crypto,0.0,0.0,67033.000000,50984,63222.00000,3811.000000,0.060280,0.960029
160,2026-08-31,IB,0.0,0.0,34250.940648,34267.147173,31278.25849,2972.682158,0.095040,0.712546
132,2026-08-31,Moomoo,0.0,0.0,112310.000000,0,90853.00000,21457.000000,0.236173,0.161481
134,2026-08-31,Tiger,0.0,0.0,222657.420000,-5992.75,193663.24000,28994.180000,0.149714,0.846533


In [13]:
# 1. Define target columns
selected_columns = [
    'Deposit_SGD', 'Withdraw_SGD', 'ending_total_assets',
    'previous_month_assets', 'P_L', 'ending_cash_balance'
]

# 2. Clean non-numeric characters and coerce to float
for col in selected_columns:
    if col in investment_summary.columns:
        investment_summary[col] = (
            investment_summary[col]
            .astype(str)
            .str.replace(',', '', regex=False)
        )
        investment_summary[col] = pd.to_numeric(investment_summary[col], errors='coerce').fillna(0)

# 3. Group by month_year on combined_summary
combined_summary = investment_summary.groupby('month_year', as_index=False)[selected_columns].sum()

# 4. Calculate total monthly return (%) on aggregated totals
combined_summary['Total_Return(%)'] = np.where(
    combined_summary['previous_month_assets'] != 0,
    (combined_summary['P_L'] / combined_summary['previous_month_assets']),
    0
)

combined_summary['Cumulative_R_total'] = (1 + (combined_summary['Total_Return(%)'])).cumprod() - 1

combined_summary.tail()

,month_year,Deposit_SGD,Withdraw_SGD,ending_total_assets,previous_month_assets,P_L,ending_cash_balance,Total_Return(%),Cumulative_R_total
41,2026-05-31,0.0,0.0,376776.614152,328629.257427,48147.356725,-60941.498748,0.146510,0.303509
42,2026-06-30,4000.0,0.0,292967.997274,376776.614152,-87808.616878,-80055.646410,-0.233052,-0.000276
43,2026-07-31,0.0,0.0,379016.498490,292967.997274,86048.501216,-91913.563116,0.293713,0.293356
44,2026-08-31,0.0,0.0,436251.360648,379016.498490,57234.862158,79258.397173,0.151009,0.488664
45,2026-09-30,0.0,0.0,222515.900000,222657.420000,-141.520000,-5382.710000,-0.000636,0.487717


In [14]:
# 1. Clean up existing columns and sort cleanly ONCE
if 'S&P' in combined_summary.columns:
    combined_summary = combined_summary.drop(columns=['S&P'])
combined_summary['month_year'] = pd.to_datetime(combined_summary['month_year'])
combined_summary = combined_summary.sort_values('month_year')

# 2. Extract accurate date ranges
start_date = combined_summary['month_year'].min().strftime('%Y-%m-%d')
end_date = (combined_summary['month_year'].max() + pd.Timedelta(days=1)).strftime('%Y-%m-%d')

# 3. Fetch S&P 500 historical data
sp500_data = yf.download('^GSPC', start=start_date, end=end_date)[['Close']].reset_index()

if isinstance(sp500_data.columns, pd.MultiIndex):
    sp500_data.columns = sp500_data.columns.get_level_values(0)

sp500_data.rename(columns={'Date': 'month_year', 'Close': 'S&P'}, inplace=True)
sp500_data['month_year'] = pd.to_datetime(sp500_data['month_year'])
sp500_data = sp500_data.sort_values('month_year')

# 🔥 FIX: Calculate S&P Returns on the isolated index data BEFORE merging.
# This prevents different brokers from mixing up your chronological shifts!
sp500_data['previous month s&p'] = sp500_data['S&P'].shift(1)
sp500_data['S&P(percent)'] = (sp500_data['S&P'] - sp500_data['previous month s&p']) / sp500_data['previous month s&p']
sp500_data['SP500_Cumulative_R'] = (1 + sp500_data['S&P(percent)'].fillna(0)).cumprod() - 1

# 4. Fetch historical Fear & Greed data from Alternative.me API
url = "https://api.alternative.me/fng/?limit=0"
response = requests.get(url).json()

fg_df = pd.DataFrame(response['data'])
fg_df['month_year'] = pd.to_datetime(fg_df['timestamp'].astype(int), unit='s')
fg_df['Fear_and_Greed_Index'] = fg_df['value'].astype(float)
fg_df = fg_df.sort_values('month_year').reset_index(drop=True)

# 5. Merge S&P data columns back into your main multi-broker summary dataframe
combined_summary = pd.merge_asof(
    combined_summary,
    sp500_data,
    on='month_year',
    direction='backward'
)

# 6. Merge Fear & Greed data columns onto your summary dataframe
combined_summary = pd.merge_asof(
    combined_summary,
    fg_df[['month_year', 'Fear_and_Greed_Index']],
    on='month_year',
    direction='backward'
)

combined_summary.columns = (
    combined_summary.columns
    .str.replace(r'[&%()\- ]', '_', regex=True) # Replace special characters with _
    .str.replace(r'_{2,}', '_', regex=True)     # Clean up any double underscores __
    .str.strip('_')                            # Strip leading/trailing underscores
)

# Display final optimized dataset
combined_summary.head()


/tmp/ipykernel_12971/4109392378.py:12: FutureWarning: YF.download() has changed argument auto_adjust default to True
  sp500_data = yf.download('^GSPC', start=start_date, end=end_date)[['Close']].reset_index()
[*********************100%***********************]  1 of 1 completed


,month_year,Deposit_SGD,Withdraw_SGD,ending_total_assets,previous_month_assets,P_L,ending_cash_balance,Total_Return,Cumulative_R_total,S_P,previous_month_s_p,S_P_percent,SP500_Cumulative_R,Fear_and_Greed_Index
0,2022-12-31,75180.0,0.0,51459.55,75180.00,-23720.45,0.79,-0.315515,-0.315515,NaN,NaN,NaN,NaN,25.0
1,2023-01-31,5000.0,0.0,60122.76,51459.55,3663.21,0.01,0.071186,-0.266790,4076.600098,4017.770020,0.014642,0.066018,51.0
2,2023-02-28,0.0,0.0,60823.08,60122.76,700.32,12491.42,0.011648,-0.258249,3970.149902,3982.239990,-0.003036,0.038181,53.0
3,2023-03-31,9000.0,0.0,69065.02,60823.08,-758.06,12582.55,-0.012463,-0.267494,4109.310059,4050.830078,0.014437,0.074571,63.0
4,2023-04-30,0.0,0.0,71066.08,69065.02,2001.06,21513.46,0.028974,-0.246270,4169.479980,4135.350098,0.008253,0.090305,60.0


In [15]:
!pip install pandas-gbq

In [ ]:
project_id = 'igneous-fort-507812-t9'       # Your actual GCP Project ID string
dataset_id = 'investment'                  # The folder ID you created in Step 1
table_id_1   = 'broker_summary'            # Table 1 name
table_id_2   = 'combined_investment_summary' # Table 2 name

destination_table_1 = f"{dataset_id}.{table_id_1}"
destination_table_2 = f"{dataset_id}.{table_id_2}"

# 1. Clean up special characters and spaces for investment_summary columns
investment_summary.columns = (
    investment_summary.columns
    .str.replace(r'[&%()\- ]', '_', regex=True)
    .str.replace(r'_{2,}', '_', regex=True)
    .str.strip('_')
)

# 2. Clean up special characters and spaces for combined_summary columns
combined_summary.columns = (
    combined_summary.columns
    .str.replace(r'[&%()\- ]', '_', regex=True)
    .str.replace(r'_{2,}', '_', regex=True)
    .str.strip('_')
)

# Stream DataFrame 1 straight to BigQuery
investment_summary.to_gbq(
    destination_table=destination_table_1,
    project_id=project_id,
    if_exists='replace',                 # Overwrites the table with updated calculations
    progress_bar=True
)

# Stream DataFrame 2 straight to BigQuery
combined_summary.to_gbq(
    destination_table=destination_table_2,
    project_id=project_id,
    if_exists='replace',                 # Overwrites the table with updated calculations
    progress_bar=True
)

# Adjusted print statement to show both success tracks
print(f"Data successfully synced to BigQuery:")
print(f" -> Table 1: {project_id}.{destination_table_1}")
print(f" -> Table 2: {project_id}.{destination_table_2}")


/tmp/ipykernel_12971/1531770586.py:26: FutureWarning: to_gbq is deprecated and will be removed in a future version. Please use pandas_gbq.to_gbq instead: https://pandas-gbq.readthedocs.io/en/latest/api.html#pandas_gbq.to_gbq
  investment_summary.to_gbq(


#### Moomoo

In [ ]:
# Specify the path to the folder holding your files
# Change 'MyDrive/YourFolder' to the actual path where your files are stored
folder_path = '/content/drive/MyDrive/Investment'

# Find all files starting with "History-Margin" in that folder
search_pattern = os.path.join(folder_path, "History-Margin*")
matching_files = glob.glob(search_pattern)

if not matching_files:
    print("No files found matching the pattern 'History-Margin'")
else:
    # 4. Sort the files by their system modification date to get the latest one
    latest_file_path_moomoo = max(matching_files, key=os.path.getmtime)

    # 5. Extract just the file name
    latest_file_name_moomoo = os.path.basename(latest_file_path_moomoo)

    print(f"Latest file path: {latest_file_path_moomoo}")
    print(f"Latest file name: {latest_file_name_moomoo}")

    moomoo_df = pd.read_csv(latest_file_path_moomoo)

In [ ]:
moomoo_df.head()

In [ ]:
# drop those which transactions were cancelled.
moomoo_df = moomoo_df[moomoo_df['Status'] != 'Cancelled']
# keep relevant columns
relevant_columns = ['Side', 'Symbol', 'Name', 'Order Price', 'Order Qty', 'Order Amount', 'Markets', 'Fill Time', 'Total']
moomoo_df = moomoo_df[relevant_columns]
moomoo_df.columns = moomoo_df.columns.str.lower()

moomoo_df.head()

In [ ]:
# extract the stock ticker from symbol
moomoo_df["stock"] = np.where(
  moomoo_df["symbol"].astype(str).str.match(r"^\d+$"),   # pure digits (HK stocks)
  moomoo_df["symbol"],                                   # keep full symbol
  moomoo_df["symbol"].astype(str).str.extract(r"^([A-Za-z]+)").iloc[:, 0]  # extract letters
)

# extract the option expiration date from Symbol
moomoo_df["expiration_date"] = moomoo_df["symbol"].str.extract(r"(\d{6})")
moomoo_df["expiration_date"] = pd.to_datetime(moomoo_df["expiration_date"], format="%y%m%d")
moomoo_df["expiration_date"] = moomoo_df["expiration_date"].dt.strftime("%d %b %y")

# identify if trade is OPT or STK and if is OPT then is it a call or put
moomoo_df["security_type"] = moomoo_df["name"].astype(str).str.contains(r"\d").map(
    {True: "OPT", False: "STK"}
)
moomoo_df["option type"] = moomoo_df["symbol"].str.extract(r"\d{6}([CP])")

moomoo_df["option type"] = moomoo_df.apply(
    lambda row: "CALL" if row["option type"] == "C" and row["security_type"] == "OPT"
    else "PUT" if row["option type"] == "P" and row["security_type"] == "OPT"
    else None,
    axis=1
)

# extract the strike price from optuons
moomoo_df["strike_raw"] = moomoo_df["symbol"].str.extract(r"[A-Za-z](\d+)$")
moomoo_df["strike_raw"] = pd.to_numeric(moomoo_df["strike_raw"], errors="coerce")
moomoo_df["strike price"] = moomoo_df["strike_raw"] / 1000

# extract and convert the order time to proper date
moomoo_df["fill time"] = moomoo_df["fill time"].str.replace(r"\s[A-Z]{2,4}$", "", regex=True)
moomoo_df["fill time"] = pd.to_datetime(
    moomoo_df["fill time"],
    format="%b %d, %Y %H:%M:%S",
    errors="coerce"
)
moomoo_df["fill time"] = moomoo_df["fill time"].dt.strftime("%d %b %y")
moomoo_df.head()


In [ ]:
# 1. Filter and make an explicit copy to avoid SettingWithCopyWarning
moomoo_stock_df = moomoo_df[moomoo_df["security_type"] == "STK"].copy()

# Clean 'order amount' string to numeric
moomoo_stock_df["order amount"] = (
    moomoo_stock_df["order amount"]
    .astype(str)
    .str.replace(r"[$,]", "", regex=True)
    .str.strip()
)
moomoo_stock_df["order amount"] = pd.to_numeric(moomoo_stock_df["order amount"], errors="coerce")

# Map transaction side to signed values
sign = moomoo_stock_df["side"].map({"Buy": 1, "Sell": -1}).fillna(0)

moomoo_stock_df["order qty"] = moomoo_stock_df["order qty"] * sign
moomoo_stock_df["order amount"] = moomoo_stock_df["order amount"] * sign

# 2. Group by stock
moomoo_stock_df = moomoo_stock_df.groupby("stock", as_index=False).agg({
    "order qty": "sum",
    "order amount": "sum",
    "total": "sum",
    "security_type": "first",
    "markets": "first"
})

# 3. Filter out zero/negative quantity FIRST to prevent division by zero
moomoo_stock_df = moomoo_stock_df[moomoo_stock_df["order qty"] >= 0].copy()

# 4. Compute average price safely
moomoo_stock_df["average_price"] = (
    moomoo_stock_df["order amount"] / moomoo_stock_df["order qty"]
)

moomoo_stock_df

In [ ]:
# 1. Get unique tickers only where order qty is not 0
active_tickers = (
    moomoo_stock_df[moomoo_stock_df["order qty"] != 0]["stock"]
    .unique()
    .tolist()
)

# 2. Fetch the latest live price from Yahoo Finance
price_data = yf.download(active_tickers, period="1d", interval="1m")["Close"]
live_prices = price_data.iloc[-1] if not price_data.empty else {}

# 3. Create the new column using np.where to check the condition
# If order qty != 0, map the ticker to its live price; otherwise, leave it as NaN (or 0)
moomoo_stock_df["live_price"] = np.where(
    moomoo_stock_df["order qty"] != 0,
    moomoo_stock_df["stock"].map(live_prices),
    np.nan,  # You can replace np.nan with 0 if preferred
)

moomoo_stock_df["realized p&l"] = np.where(
    moomoo_stock_df["order qty"] == 0,
    (moomoo_stock_df["order amount"] * -1) - moomoo_stock_df["total"],  # <-- Added comma here
    np.nan,
)

# Calculate Unrealised P&L where order qty > 0
moomoo_stock_df["unrealized p&l"] = np.where(
    moomoo_stock_df["order qty"] > 0,
    (moomoo_stock_df["live_price"] - moomoo_stock_df["average_price"])
    * moomoo_stock_df["order qty"],
    np.nan,  # Fills closed positions (qty <= 0) with NaN
)

moomoo_stock_df

In [ ]:
moomoo_option_df = moomoo_df[moomoo_df["security_type"] == "OPT"]
moomoo_option_df['order amount'] = pd.to_numeric(moomoo_option_df['order amount'])
moomoo_option_df['total'] = pd.to_numeric(moomoo_option_df['total'])
moomoo_option_df['pl'] = moomoo_option_df['order amount'] - moomoo_option_df['total']
# Rename 'markets' to 'market' for consistency with tiger_option_df
moomoo_option_df = moomoo_option_df.drop(columns=[
    'side', 'symbol', 'name', 'order price', 'strike_raw', 'order amount', 'total'])
moomoo_option_df.head()

### Tiger

In [ ]:
search_pattern = os.path.join(folder_path, "STOCK_P&L*")
matching_files = glob.glob(search_pattern)

if not matching_files:
    print("No files found matching the pattern 'STOCK_P&L'")
else:
    # 4. Sort the files by their system modification date to isolate the latest one
    latest_file_path_stock = max(matching_files, key=os.path.getmtime)

    # 5. Extract just the file name string
    latest_file_name_stock = os.path.basename(latest_file_path_stock)

    print(f"Latest file path: {latest_file_path_stock}")
    print(f"Latest file name: {latest_file_name_stock}")

    # 6. Extract using read_excel instead of read_csv
    tiger_df = pd.read_excel(latest_file_path_stock)
tiger_df

In [ ]:
tiger_df.columns = tiger_df.columns.str.lower()
# extract stock
tiger_df["stock"] = tiger_df["symbol"].str.extract(r"^([^\d]*)")

# extract expiration date
tiger_df["expiration_date"] = tiger_df["symbol"].str.extract(r"(\d{8})")
tiger_df["expiration_date"] = pd.to_datetime(tiger_df["expiration_date"], format="%Y%m%d")
tiger_df["expiration_date"] = tiger_df["expiration_date"].dt.strftime("%d %b %y")

today = pd.Timestamp.today()

tiger_df["fill time"] = today - pd.to_timedelta(tiger_df["position days"], unit="D")
tiger_df["fill time"] = tiger_df["fill time"].dt.strftime("%d %b %y")
tiger_df["option type"] = tiger_df["symbol"].str.extract(r"\b(PUT|CALL)\b")

tiger_df["option type"] = tiger_df.apply(
    lambda row: "CALL" if row["option type"] == "CALL" and row["security type"] == "OPT"
    else "PUT" if row["option type"] == "PUT" and row["security type"] == "OPT"
    else None,
    axis=1
)

tiger_df["strike price"] = tiger_df["symbol"].str.split().str[-1]
tiger_df["strike price"] = pd.to_numeric(tiger_df["strike price"], errors="coerce")

tiger_df.rename(columns={'market': 'markets'}, inplace=True)

tiger_df.head()

In [ ]:
tiger_option_df = tiger_df[tiger_df["security type"] == "OPT"]

tiger_option_df['pl'] = np.where(tiger_option_df['unrealized p&l'] == 0, tiger_option_df['realized p&l'], tiger_option_df['unrealized p&l'])
# Rename 'security type' to 'security_type' for consistency with moomoo_option_df
tiger_option_df.rename(columns={'security type': 'security_type'}, inplace=True)
tiger_option_df = tiger_option_df.drop(columns=[
    'symbol', 'name', 'currency', 'total p&l', 'dividends',
    'position days','unrealized p&l', 'realized p&l'])
tiger_option_df.rename(columns={'transactions': 'order qty'}, inplace=True)
tiger_option_df.head()

In [ ]:
tiger_stock_df = tiger_df[tiger_df["security type"] == "STK"]

# 1. Filter for rows where dividends are greater than 0 to create the duplicates
div_rows = tiger_stock_df[tiger_stock_df["dividends"] > 0].copy()

# 2. Modify the values on the duplicated rows
div_rows["realized p&l"] = div_rows["dividends"]
div_rows["unrealized p&l"] = 0
div_rows["security type"] = "DIV"

# 3. Append the duplicated rows back to the original DataFrame
tiger_stock_df = pd.concat([tiger_stock_df, div_rows], ignore_index=True)
tiger_stock_df = tiger_stock_df.drop(columns=[
    'symbol', 'name', 'currency', 'total p&l',
    'position days', 'transactions', 'expiration_date',
    'fill time', 'option type', 'strike price', 'dividends'])

# Rename 'security type' to 'security_type' for consistency
tiger_stock_df.rename(columns={'security type': 'security_type'}, inplace=True)
# Rename P&L columns to be consistent
tiger_stock_df.rename(columns={'unrealized p&l': 'unrealized p&l', 'realized p&l': 'realized p&l'}, inplace=True)

tiger_stock_df.head()

In [ ]:
tiger_stock_df.info()

##### IB

In [ ]:
ib_stock_df = trading_df[trading_df['AssetClass'] == 'STK']
ib_stock_df

In [ ]:
# --- Placeholder function to fetch live prices ---
# Replace this function or logic with your actual API / data source (e.g., yfinance)
def get_live_price(symbol):
    # Simulated current prices for demonstration purposes
    live_prices = {'KO': 65.50, 'LULU': 225.00, 'MSFT': 420.00, 'NVDA': 145.00}
    return live_prices.get(symbol, 0.0)

# --- Clean hidden formatting variables ---
ib_stock_df.columns = ib_stock_df.columns.str.strip().str.replace(r'[^\w]', '', regex=True)
ib_stock_df['DateTime'] = pd.to_datetime(ib_stock_df['DateTime'], format='%Y%m%d;%H%M%S')
ib_stock_df = ib_stock_df.sort_values(by='DateTime').reset_index(drop=True)

# 2. Process transactions row-by-row
portfolio = {}

for index, row in ib_stock_df.iterrows():
    symbol = row['Symbol']
    qty = row['Quantity']
    price = row['ClosePrice']

    # Track the extra metadata columns here
    currency = row['CurrencyPrimary']
    asset_class = row['AssetClass']

    if symbol not in portfolio:
        portfolio[symbol] = {
            'shares': 0.0,
            'avg_price': 0.0,
            'realized_pnl': 0.0,         # Renamed to strictly track closed-out trade profits
            'markets': currency,
            'security_type': asset_class
        }

    state = portfolio[symbol]

    if qty > 0:  # BUY
        total_cost = (state['shares'] * state['avg_price']) + (qty * price)
        state['shares'] += qty
        state['avg_price'] = total_cost / state['shares']

    elif qty < 0:  # SELL
        sell_qty = abs(qty)
        # Realized P/L only builds up when selling shares
        profit = (price - state['avg_price']) * sell_qty
        state['realized_pnl'] += profit
        state['shares'] -= sell_qty

        # Reset cost basis if position hits zero
        if state['shares'] <= 0:
            state['shares'] = 0.0
            state['avg_price'] = 0.0

# 3. GENERATE COLUMNS DYNAMICALLY WITH LIVE PRICE AND TRUE UNREALIZED P/L
summary_rows = []
for symbol, metrics in portfolio.items():
    qty = metrics['shares']
    avg_price = metrics['avg_price']
    realized_amt = metrics['realized_pnl']

    # Initialize variables for live calculations
    live_price = 0.0
    unrealized_pnl = 0.0

    # If we still hold shares, calculate true live unrealized P/L
    if qty > 0:
        live_price = get_live_price(symbol)
        unrealized_pnl = (live_price - avg_price) * qty

    summary_rows.append({
        'stock': symbol,
        'order qty': qty,
        'average_price': round(avg_price, 2),
        # UPDATED: All currency values below are now pure floats, no dollar signs
        'live_price': round(live_price, 2) if qty > 0 else 0.0,
        'realized p&l': round(realized_amt, 2),
        'unrealized p&l': round(unrealized_pnl, 2),
        'markets': metrics['markets'],
        'security_type': metrics['security_type']
    })

ib_stock_summary_df = pd.DataFrame(summary_rows)
ib_stock_summary_df



In [ ]:
ib_option_df = trading_df[trading_df['AssetClass'] == 'OPT']
ib_option_df.tail()

In [ ]:
# extract the stock ticker from symbol
ib_option_df["stock"] = np.where(
  ib_option_df["Symbol"].astype(str).str.match(r"^\d+$"),   # pure digits (HK stocks)
  ib_option_df["Symbol"],                                   # keep full symbol
  ib_option_df["Symbol"].astype(str).str.extract(r"^([A-Za-z]+)").iloc[:, 0]  # extract letters
)
# extract the option expiration date from Symbol
ib_option_df["expiration_date"] = ib_option_df["Symbol"].str.extract(r"(\d{6})")
ib_option_df["expiration_date"] = pd.to_datetime(ib_option_df["expiration_date"], format="%y%m%d", errors='coerce')
ib_option_df["expiration_date"] = ib_option_df["expiration_date"].dt.strftime("%d %b %y")

# extract the strike price from optuons
ib_option_df["strike_raw"] = ib_option_df["Symbol"].str.extract(r"[A-Za-z](\d+)$")
ib_option_df["strike_raw"] = pd.to_numeric(ib_option_df["strike_raw"], errors="coerce")
ib_option_df["strike price"] = ib_option_df["strike_raw"] / 1000

ib_option_df['order qty'] = ib_option_df['Quantity'].abs()

# 1. Extract the 8 consecutive digits (YYYYMMDD)
ib_option_df["fill time"] = ib_option_df["DateTime"].str.extract(r"(\d{8})")

# 2. Convert to datetime using uppercase %Y for 4-digit years
ib_option_df["fill time"] = pd.to_datetime(ib_option_df["fill time"], format="%Y%m%d", errors='coerce')

# 3. Format to "DD Mmm YY" (e.g., 31 Oct 25)
ib_option_df["fill time"] = ib_option_df["fill time"].dt.strftime("%d %b %y")

# Rename specific columns in-place or assign to DataFrame
ib_option_df = ib_option_df.rename(columns={
    'CurrencyPrimary': 'markets',
    'AssetClass': 'security_type',
    'Put/Call' : 'option type',
    'FifoPnlRealized': 'pl'
})

ib_option_df = ib_option_df.drop(columns=['Symbol', 'Description', 'Expiry', 'DateTime', 'strike_raw', 'Quantity', 'ClosePrice'])
ib_option_df


In [ ]:
# 1. Extract rows where Description contains 'DIVIDEND'
ib_div_df = div_df[
    div_df['Description'].str.contains('DIVIDEND', case=False, na=False)
].copy() # Using .copy() avoids a hidden SettingWithCopyWarning

# 2. Drop columns you don't need
# Note: Removed 'Put/Call' from the drop list since you are renaming it right after!
ib_div_df = ib_div_df.drop(columns=['Description', 'Expiry', 'Date/Time', 'DividendType', 'Type'])

# 3. Rename columns to your standardized format
ib_div_df = ib_div_df.rename(columns={
    'CurrencyPrimary': 'markets',
    'AssetClass': 'security_type',
    'Put/Call' : 'option_type',
    'Amount': 'realized p&l',
    'Symbol': 'stock'
})

# 4. Set security type explicitly to 'DIV'
ib_div_df['security_type'] = 'DIV'

# 5. Group by stock, SUM the math column, and KEEP the first text string for others
agg_rules = {
    'realized p&l': 'sum',
    'markets': 'first',
    'security_type': 'first'
}

ib_div_df = ib_div_df.groupby('stock').agg(agg_rules).reset_index()

# 6. Sort by the highest profit
ib_div_df = ib_div_df.sort_values(by='realized p&l', ascending=False)

# 7. View the perfectly aggregated summary table
ib_div_df.head()


##### Combine the stock df from all investment brokers

In [ ]:
# 1. Add the "Brokers" column to each DataFrame
moomoo_stock_df['Brokers'] = 'Moomoo'
tiger_stock_df['Brokers'] = 'Tiger'
ib_stock_summary_df['Brokers'] = 'IB'
ib_div_df['Brokers'] = 'IB'

# 1.5 Cleanly reset the row indices to prevent internal alignment errors
moomoo_stock_df = moomoo_stock_df.reset_index(drop=True)
tiger_stock_df = tiger_stock_df.reset_index(drop=True)
ib_stock_summary_df = ib_stock_summary_df.reset_index(drop=True)

# 2. Stack the DataFrames vertically (This will now run smoothly!)
combined_stock_df = pd.concat([moomoo_stock_df, tiger_stock_df, ib_stock_summary_df, ib_div_df], ignore_index=True)

# 3. Clean column names
combined_stock_df.columns = (
    combined_stock_df.columns
    .str.replace(r'[&%()\- ]', '_', regex=True)
    .str.replace(r'_{2,}', '_', regex=True)
    .str.strip('_')
)

# 4. Fill NaNs and calculate P&L
cols_to_fill = ['realized_p_l', 'unrealized_p_l']
combined_stock_df[cols_to_fill] = combined_stock_df[cols_to_fill].fillna(0)
combined_stock_df['pl'] = combined_stock_df['unrealized_p_l'] + combined_stock_df['realized_p_l']

combined_stock_df.head()


##### Combine the option df from all investment brokers

In [ ]:
# 1. Add the "Brokers" column to each DataFrame
moomoo_option_df['Brokers'] = 'Moomoo'
tiger_option_df['Brokers'] = 'Tiger'
ib_option_df['Brokers'] = 'IB'

# 2. Stack the DataFrames vertically, matching column names
combined_option_df = pd.concat([moomoo_option_df, tiger_option_df, ib_option_df], ignore_index=True)
combined_option_df.columns = (
    combined_option_df.columns
    .str.replace(r'[&%()\- ]', '_', regex=True) # Replace special characters with _
    .str.replace(r'_{2,}', '_', regex=True)     # Clean up any double underscores __
    .str.strip('_')                            # Strip leading/trailing underscores
)
combined_option_df

In [ ]:
# 1. Convert month_year to datetime to allow relative date comparison
# (Format argument is auto-inferred; adjust if yours is custom, e.g., format='%b %Y')
investment_summary['month_year_dt'] = pd.to_datetime(investment_summary['month_year'])

# 2. Get the first day of the current month
first_of_this_month = pd.Timestamp.now().replace(day=1, hour=0, minute=0, second=0, microsecond=0)

# 3. Filter for Crypto, non-zero P&L, and month_year strictly before current month (< first_of_this_month)
crypto_pl_df = investment_summary[
    (investment_summary['Brokers'].str.contains('Crypto', case=False, na=False)) &
    (investment_summary['P_L'] != 0) &
    (investment_summary['month_year_dt'] < first_of_this_month)
][['month_year', 'Brokers', 'P_L']]

# View the extracted columns
crypto_pl_df

In [ ]:
combined_df = pd.concat([combined_option_df, combined_stock_df], ignore_index=True)

# Ensure 'realized_p_l', 'unrealized_p_l', and 'pl' are numeric
# These columns might contain string values (e.g., from ib_stock_summary_df) which prevents multiplication.
numeric_cols_to_clean = ['realized_p_l', 'unrealized_p_l', 'pl']
for col in numeric_cols_to_clean:
    if col in combined_df.columns:
        combined_df[col] = combined_df[col].astype(str).str.replace(r'[$,]', '', regex=True)
        combined_df[col] = pd.to_numeric(combined_df[col], errors='coerce').fillna(0)

# Fetch live currency rates against SGD
usd_sgd = yf.Ticker("USDSGD=X").history(period="1d")['Close'].iloc[-1]
hkd_sgd = yf.Ticker("HKDSGD=X").history(period="1d")['Close'].iloc[-1]

# 2. Define conditions based on the market column
conditions = [
    combined_df['markets'].str.upper().str.contains('US|USD', na=False),
    combined_df['markets'].str.upper().str.contains('HK|HKD', na=False),
    combined_df['markets'].str.upper().str.contains('SG|SGD', na=False)
]

choices = [
    usd_sgd,
    hkd_sgd,
    1.0
]

# 3. Create the FX rate column and calculate SGD value
combined_df['fx_rate'] = np.select(conditions, choices, default=1.0)
combined_df['realized_p_l'] = combined_df['realized_p_l'] * combined_df['fx_rate']
combined_df['unrealized_p_l'] = combined_df['unrealized_p_l'] * combined_df['fx_rate']
combined_df['pl'] = combined_df['pl'] * combined_df['fx_rate']
combined_df['stock'] = combined_df['stock'].str.strip().str.upper()

# Sector mapping lookup
SECTOR_LOOKUP = {
    # Main Tech Sector Names
    'Information Technology': 'Tech',
    'Technology': 'Tech',                           # Common alternative API naming
    'Semiconductors': 'Tech',                        # Sometimes returned as sector/industry
    'Software': 'Tech',

    # Communication
    'Communication Services': 'Communication',
    'Telecommunications': 'Communication',           # Older/alternative provider label
    'Media & Entertainment': 'Communication',

    # Consumer
    'Consumer Cyclical': 'Consumer',
    'Consumer Defensive': 'Consumer',
    'Consumer Discretionary': 'Consumer',           # Official GICS name for Cyclical
    'Consumer Staples': 'Consumer',                 # Official GICS name for Defensive

    # Financials
    'Financial Services': 'Financial',
    'Financials': 'Financial',
    'Banking': 'Financial',

    # Commodities & Energy
    'Basic Materials': 'Commodity',
    'Materials': 'Commodity',                        # Shortened GICS name
    'Energy': 'Commodity',
    'Utilities': 'Commodity'                        # Often grouped with hard assets/commodities
}

def get_stock_category(ticker):
    try:
        # Handle Gold/Commodity ETFs directly by symbol pattern
        if ticker in ['IAU', 'GLD', 'SLV', 'USO']:
            return 'Commodity'

        info = yf.Ticker(ticker).info
        sector = info.get('sector', '')
        quote_type = info.get('quoteType', '')

        # Crypto check
        if quote_type == 'CRYPTOCURRENCY' or 'Crypto' in sector or str(ticker).endswith('-USD'):
            return 'Crypto'

        # Return mapped sector or default to 'Other'
        return SECTOR_LOOKUP.get(sector, 'Other')

    except Exception:
        return 'Other'

# 1. Extract only unique tickers (dropna removes null values if any exist)
unique_stocks = combined_df['stock'].dropna().drop_duplicates().tolist()

# 2. Fetch categories ONLY for unique tickers (Fast: runs yfinance once per unique ticker)
stock_type_mapping = {ticker: get_stock_category(ticker) for ticker in unique_stocks}

# 3. Map back to the full DataFrame instantly
combined_df['stock_type'] = combined_df['stock'].map(stock_type_mapping).fillna('Other')
combined_df.head()

In [ ]:
# 1. Create a formatted DataFrame aligned with combined_df structure
crypto_formatted = pd.DataFrame(index=crypto_pl_df.index)

# 2. Map existing crypto columns to combined_df schema
# 'Brokers' maps to 'Brokers', 'security_type', and 'stock_type'
crypto_formatted['Brokers'] = crypto_pl_df['Brokers']
crypto_formatted['security_type'] = crypto_pl_df['Brokers']
crypto_formatted['stock_type'] = crypto_pl_df['Brokers']
crypto_formatted['stock'] = crypto_pl_df['Brokers']

# 'P_L' maps to 'pl'
crypto_formatted['pl'] = crypto_pl_df['P_L']

# 3. Fill remaining combined_df columns with NaN
for col in combined_df.columns:
    if col not in crypto_formatted.columns:
        crypto_formatted[col] = np.nan

# Reorder columns to match combined_df exactly
crypto_formatted = crypto_formatted[combined_df.columns]

# 4. Concatenate into combined_df
combined_df = pd.concat([combined_df, crypto_formatted], ignore_index=True)

combined_df['option_type'] = combined_df['option_type'].map({'PUT': 'P', 'CALL': 'C'})
combined_df = combined_df.drop(columns=['fx_rate', 'total', 'order_amount'])
combined_df.head()

In [ ]:
# Extract all columns where unrealized_p_l != 0 AND security_type is 'DIV'
div_non_zero_df = combined_df[
    (combined_df['unrealized_p_l'] != 0) &
    (combined_df['security_type'] == 'DIV')
]

# View the full filtered DataFrame
div_non_zero_df

In [ ]:
table_id_3 = 'trading'

destination_table_3 = f"{dataset_id}.{table_id_3}"


# Stream DataFrame 1 straight to BigQuery
combined_df.to_gbq(
    destination_table=destination_table_3,
    project_id=project_id,
    if_exists='replace',                 # Overwrites the table with updated calculations
    progress_bar=True
)
# Cleaned print output
print("Data successfully synced to BigQuery:")
print(f" -> Table 3: {destination_table_3}")

In [ ]:
# Copy your notebook to the active working directory
!cp "/content/drive/MyDrive/Colab Notebooks/Investment.ipynb" ./ 2>/dev/null || true

In [ ]:
!git add .
!git commit -m "Add Investment notebook"
!git push -u origin main --force